# Phần Đức: Feature Engineering B và Decision Tree A/B

Dùng nguyên các file raw và nhãn do Mạnh chia: 5.634 Train, 1.409 Test. Không chia lại hoặc đổi thứ tự dòng. Notebook này kiểm tra và trình bày kết quả đã chạy từ CLI; bật `RUN_TRAINING` để huấn luyện lại toàn bộ.

**Giới hạn thí nghiệm:** Test đã được xem trong các lần thử trước (B 67 cột rồi B 29 cột). Không mô tả Test là chưa từng được xem. B 29 cột được giữ cố định trong lần sửa này; mọi GridSearch chạy trên Train. Không thay đặc trưng theo điểm Test mới.

In [1]:
from pathlib import Path
import sys, json, csv
import numpy as np
import joblib

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src/config.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'src/config.py').exists()
sys.path.insert(0, str(PROJECT_ROOT))
from src.feature_engineering_b import read_raw_csv
from src.train_decision_tree import load_numeric_matrix, load_target
print('Python:', sys.version.split()[0])

Python: 3.12.10


## Đặc trưng B

Giữ 4 biến số: SeniorCitizen, tenure, MonthlyCharges, TotalCharges. Giữ 8 biến phân loại: Partner, Dependents, InternetService, OnlineSecurity, TechSupport, Contract, PaperlessBilling, PaymentMethod. Dùng đủ one-hot của Train.

Thêm 3 tương tác dựa trên EDA của Mạnh:
- MonthToMonthFiber: hợp đồng tháng và Fiber optic.
- MonthToMonthElectronicCheck: hợp đồng tháng và Electronic check.
- RiskServiceGap: số dịch vụ OnlineSecurity/TechSupport có trạng thái No (0–2). No internet service không được tính là thiếu hỗ trợ.

Tổng: 29 cột. Không chuẩn hóa B vì Decision Tree không cần; Hải cần fit scaler trong pipeline CV của K-NN. B_core (26 cột) chỉ dùng đối chiếu đóng góp của 3 tương tác trên Train, không phải kết quả thứ ba gửi Hùng.

In [2]:
RUN_TRAINING = False  # True: chạy lại 3 x 120 cấu hình x 5 fold, có thể mất vài phút.
if RUN_TRAINING:
    from src.train_decision_tree import main
    main(n_jobs=2)
else:
    print('Đọc và xác minh kết quả đã chạy bằng python -m src.train_decision_tree.')

Đọc và xác minh kết quả đã chạy bằng python -m src.train_decision_tree.


## Cross-validation đúng phạm vi

Mỗi fold clone pipeline gồm tiền xử lý và DecisionTreeClassifier. Điền thiếu (median/mode), mức one-hot và scaler A chỉ học từ phần training của fold đó. GridSearchCV 5-fold phân tầng, random_state=42, tối ưu F1 của Churn=Yes. Bảng so sánh dưới đây là kết quả CV để tham khảo, không phải ước lượng độc lập sau chọn mô hình.

Sau tìm tham số, quy tắc fit toàn Train được đối chiếu với CSV A/B; cây cuối fit trên chính các CSV này. Mô hình pipeline nhận raw và mô hình cây nhận CSV phải dự đoán giống nhau. File A/raw/nhãn của Mạnh được giữ nguyên.

In [3]:
report = PROJECT_ROOT / 'report'
comparison = json.loads((report / 'duc_feature_comparison.json').read_text(encoding='utf-8'))
for row in comparison['rows']:
    print(row)
manifest = json.loads((report / 'duc_run_manifest.json').read_text(encoding='utf-8'))
print('Môi trường:', manifest['versions'])

{'flow': 'A', 'features': 45, 'cv_f1': 0.6209339280993824, 'best_params': {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 8, 'min_samples_leaf': 40}}
{'flow': 'B_core', 'features': 26, 'cv_f1': 0.6221531481134194, 'best_params': {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 8, 'min_samples_leaf': 40}}
{'flow': 'B', 'features': 29, 'cv_f1': 0.6262255391903124, 'best_params': {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 40}}
Môi trường: {'python': '3.12.10', 'scikit_learn': '1.9.0', 'numpy': '2.5.0', 'scipy': '1.18.0', 'joblib': '1.5.3'}


In [4]:
processed = PROJECT_ROOT / 'data/processed'
transformer = joblib.load(PROJECT_ROOT / 'models/feature_engineer_B.pkl')
for split, expected_rows in [('train',5634), ('test',1409)]:
    raw, _ = read_raw_csv(processed / f'X_{split}_raw.csv')
    X, names = load_numeric_matrix(processed / f'X_{split}_B.csv')
    y = load_target(processed / f'y_{split}.csv')
    assert len(raw) == len(X) == len(y) == expected_rows
    assert names == transformer.feature_names_
    np.testing.assert_array_equal(X, transformer.transform(raw))
    print(split, X.shape, 'đúng dữ liệu và thứ tự dòng')

train

 (5634, 29) đúng dữ liệu và thứ tự dòng


test (1409, 29) đúng dữ liệu và thứ tự dòng


In [5]:
from src.evaluate import danh_gia_mo_hinh
raw_test, _ = read_raw_csv(processed / 'X_test_raw.csv')
y_test = load_target(processed / 'y_test.csv')
results = []
for flow in ('A','B'):
    X, _ = load_numeric_matrix(processed / f'X_test_{flow}.csv')
    tree = joblib.load(PROJECT_ROOT / f'models/decision_tree_{flow}.pkl')
    pipeline = joblib.load(PROJECT_ROOT / f'models/decision_tree_{flow}_pipeline.pkl')
    predictions = tree.predict(X)
    np.testing.assert_array_equal(predictions, pipeline.predict(raw_test))
    metrics = danh_gia_mo_hinh(y_test, predictions, in_chi_tiet=False)
    saved = json.loads((PROJECT_ROOT / f'results/duc_decision_tree_{flow}.json').read_text(encoding='utf-8'))
    for name in ('accuracy','precision','recall','f1_score','confusion_matrix'):
        assert saved[name] == metrics[name]
    results.append(saved)
    print(flow, json.dumps(metrics, ensure_ascii=False))
print('Ma trận: hàng = thực tế, cột = dự đoán, thứ tự [No, Yes].')

A {"accuracy": 0.7615330021291696, "precision": 0.5358490566037736, "recall": 0.7593582887700535, "f1_score": 0.6283185840707964, "confusion_matrix": [[789, 246], [90, 284]], "positive_label": "Yes"}
B {"accuracy": 0.7459190915542938, "precision": 0.5141843971631206, "recall": 0.7754010695187166, "f1_score": 0.6183368869936035, "confusion_matrix": [[761, 274], [84, 290]], "positive_label": "Yes"}
Ma trận: hàng = thực tế, cột = dự đoán, thứ tự [No, Yes].


In [6]:
best_f1 = max(results, key=lambda r: r['f1_score'])
best_recall = max(results, key=lambda r: r['recall'])
print('F1 Test cao nhất trong hai cây:', best_f1['flow'])
print('Recall Test cao nhất trong hai cây:', best_recall['flow'])
print('Gửi cả hai kết quả cho Hùng. Việc chọn mô hình chung còn cần K-NN của Hải.')

F1 Test cao nhất trong hai cây: A
Recall Test cao nhất trong hai cây: B
Gửi cả hai kết quả cho Hùng. Việc chọn mô hình chung còn cần K-NN của Hải.


## Chạy kiểm tra đầy đủ

Từ thư mục gốc, dùng môi trường cài từ requirements.txt (Python 3.12):

```bash
python -m unittest discover -s tests -v
```

Bộ kiểm tra gồm: đối chiếu bộ xử lý A gốc; công thức tương tác B; thứ tự dòng/nhãn với dữ liệu nguồn; unknown/missing/invalid; học riêng từng fold; CLI tạo pickle rồi tiến trình mới nạp lại; dự đoán raw/CSV; tính metric thủ công; kiểm tra hash và bảng CV. Các artifact đang bàn giao được mô tả ở report/duc_review.md.